# Bays (2014) Figure 2 — GP Model Predictions

Panels **c** (error distributions), **d** (variance vs set size), **e** (deviation from normal), **f** (kurtosis).

Uses the **multi-location factorised decoder** (Eqs. 23-28) with marginalisation over non-cued items.

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = str(Path.cwd().parents[1])
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors
import time
from scipy.special import logsumexp

from core.encoder.gaussian_process import (
    generate_neuron_population, periodic_rbf_kernel, sample_gp_function,
)
from core.encoder.divisive_normalization import dn_pointwise
from core.encoder.poisson_spike import generate_spikes

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
# Single-location full Poisson ML decoder + circular stats
# (Not in core — core has the multi-item factorised decoder)

def compute_log_likelihood(counts, g, T_d):
    log_g = np.log(np.maximum(g, 1e-30))
    return counts @ log_g - T_d * np.sum(g, axis=0)

def compute_circular_error(theta_true, theta_hat):
    return np.angle(np.exp(1j * (theta_hat - theta_true)))

def circular_variance(errors):
    return 1.0 - np.abs(np.mean(np.exp(1j * errors)))

def circular_kurtosis(errors):
    V = circular_variance(errors)
    rho2 = np.abs(np.mean(np.exp(2j * errors)))
    kappa2 = 1.0 - rho2
    return kappa2 / max(V**2, 1e-15) if V > 1e-10 else 0.0

def compute_deviation_from_normal(errors, n_bins=50):
    from scipy.stats import vonmises
    bin_edges = np.linspace(-np.pi, np.pi, n_bins + 1)
    centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    emp, _ = np.histogram(errors, bins=bin_edges, density=True)
    V = circular_variance(errors)
    kappa_fit = max(0.01, 1.0 / V - 1) if V > 0.01 else 100.0
    vm_pdf = vonmises.pdf(centers, kappa_fit)
    return {'bin_centers': centers, 'empirical': emp, 'normal_fit': vm_pdf,
            'deviation': emp - vm_pdf}

# Multi-item factorised decoder (Eqs. 23-26)
def compute_spike_weighted_log_tuning(counts, f_list):
    return [counts @ f_k for f_k in f_list]

def compute_marginal_log_likelihood_efficient(L_list, cued_idx):
    ll = L_list[cued_idx].copy()
    for k in range(len(L_list)):
        if k != cued_idx:
            ll = ll + logsumexp(L_list[k])
    return ll

In [ ]:
def generate_population(M, n_theta, lengthscale, n_locations=1, seed=42):
    population = generate_neuron_population(
        n_neurons=M, n_orientations=n_theta, n_locations=n_locations,
        base_lengthscale=lengthscale, lengthscale_variability=0.0, seed=seed)
    thetas = population[0]['orientations']
    f_all = []
    for loc in range(n_locations):
        f_loc = np.array([population[n]['f_samples'][loc, :] for n in range(M)])
        f_all.append(f_loc)
    return thetas, f_all

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================
M = 100; N_THETA = 64; N_TRIALS = 5000; T_D = 0.1; SIGMA_SQ = 1e-6
LAMBDA_BASE = 0.5; GAMMA = 100.0; SET_SIZES = [1, 2, 4, 8]
SEED = 42; N_SEEDS = 3; N_BINS = 50

## Multi-location trial engine

For each trial at set size l: sample l orientations, compute pre-normalised response as product across locations, apply DN, generate Poisson spikes, decode via factorised marginalisation.

In [ ]:
def run_multiloc_trials(f_all, thetas, active_locs, cued_index,
                       gamma, T_d, sigma_sq, n_trials, rng):
    l = len(active_locs)
    M, n_theta = f_all[0].shape
    errors = np.empty(n_trials)
    f_active = [f_all[loc] for loc in active_locs]

    for t in range(n_trials):
        theta_indices = rng.randint(n_theta, size=l)
        log_r_pre = np.zeros(M)
        for k in range(l):
            log_r_pre += f_active[k][:, theta_indices[k]]
        r_pre = np.exp(log_r_pre)
        rates = dn_pointwise(r_pre, gamma, sigma_sq)
        counts = generate_spikes(rates, T_d, rng)
        L_list = compute_spike_weighted_log_tuning(counts, f_active)
        ll_marginal = compute_marginal_log_likelihood_efficient(L_list, cued_index)
        idx_hat = np.argmax(ll_marginal)
        errors[t] = compute_circular_error(thetas[theta_indices[cued_index]], thetas[idx_hat])
    return errors

## Sweep set sizes

In [ ]:
t0 = time.time()
max_locs = max(SET_SIZES)
all_seeds = []

for s in range(N_SEEDS):
    cseed = SEED + s * 1000
    thetas, f_all = generate_population(M, N_THETA, LAMBDA_BASE, max_locs, cseed)
    seed_data = {}
    for N in SET_SIZES:
        rng = np.random.RandomState(cseed + N)
        errors = run_multiloc_trials(f_all, thetas, tuple(range(N)), 0,
                                      GAMMA, T_D, SIGMA_SQ, N_TRIALS, rng)
        seed_data[N] = {
            'errors': errors,
            'variance': circular_variance(errors),
            'kurtosis': circular_kurtosis(errors),
            'deviation': compute_deviation_from_normal(errors, N_BINS),
        }
        print(f"  seed={s} N={N}: var={seed_data[N]['variance']:.4f}")
    all_seeds.append(seed_data)

# Aggregate
summary = {}
for N in SET_SIZES:
    vs = [sd[N]['variance'] for sd in all_seeds]
    ks = [sd[N]['kurtosis'] for sd in all_seeds]
    emps = np.array([sd[N]['deviation']['empirical'] for sd in all_seeds])
    devs = np.array([sd[N]['deviation']['deviation'] for sd in all_seeds])
    summary[N] = {
        'var_mean': np.mean(vs), 'var_se': np.std(vs,ddof=1)/np.sqrt(N_SEEDS) if N_SEEDS>1 else 0,
        'kurt_mean': np.mean(ks), 'kurt_se': np.std(ks,ddof=1)/np.sqrt(N_SEEDS) if N_SEEDS>1 else 0,
        'emp_mean': np.mean(emps,axis=0), 'dev_mean': np.mean(devs,axis=0),
        'dev_se': np.std(devs,axis=0,ddof=1)/np.sqrt(N_SEEDS) if N_SEEDS>1 else np.zeros_like(devs[0]),
    }
bins = all_seeds[0][SET_SIZES[0]]['deviation']['bin_centers']
print(f"Done in {time.time()-t0:.1f}s")

## Plot: 2 rows x 5 columns

In [ ]:
RED = '#CC2222'
fig = plt.figure(figsize=(15, 6.5))
gs = gridspec.GridSpec(2, len(SET_SIZES)+1, width_ratios=[1]*len(SET_SIZES)+[1.3],
                       hspace=0.4, wspace=0.35)

for i, N in enumerate(SET_SIZES):
    ax = fig.add_subplot(gs[0, i])
    ax.plot(bins, summary[N]['emp_mean'], color=RED, lw=1.5)
    ax.set_xlim(-np.pi, np.pi); ax.set_title(f'{N}', fontweight='bold')
    ax.set_xticks([-np.pi,0,np.pi]); ax.set_xticklabels([r'$-\pi$','0',r'$\pi$'])
    if i==0: ax.set_ylabel('density')

ax_v = fig.add_subplot(gs[0, len(SET_SIZES)])
ns = np.array(SET_SIZES, dtype=float)
vm = [summary[N]['var_mean'] for N in SET_SIZES]
ax_v.plot(ns, vm, 'o-', color=RED, lw=1.5); ax_v.set_xscale('log',base=2); ax_v.set_yscale('log',base=2)
ax_v.set_xticks(SET_SIZES); ax_v.set_xticklabels([str(n) for n in SET_SIZES])
ax_v.set_xlabel('items'); ax_v.set_ylabel('variance')
ax_v.text(-0.18,1.08,r'$\mathbf{d}$',transform=ax_v.transAxes,fontsize=15,fontweight='bold',va='top')

for i, N in enumerate(SET_SIZES):
    ax = fig.add_subplot(gs[1, i])
    ax.plot(bins, summary[N]['dev_mean'], color=RED, lw=1.5)
    ax.fill_between(bins, summary[N]['dev_mean']-summary[N]['dev_se'],
                     summary[N]['dev_mean']+summary[N]['dev_se'], color=RED, alpha=0.12)
    ax.axhline(0, color='gray', lw=0.5); ax.set_xlim(-np.pi, np.pi)
    ax.set_xticks([-np.pi,0,np.pi]); ax.set_xticklabels([r'$-\pi$','0',r'$\pi$'])
    if i==0: ax.set_ylabel(r'$\Delta$ density')

ax_k = fig.add_subplot(gs[1, len(SET_SIZES)])
km = [summary[N]['kurt_mean'] for N in SET_SIZES]
ax_k.plot(ns, km, 'o-', color=RED, lw=1.5); ax_k.set_xscale('log',base=2); ax_k.set_yscale('log',base=2)
ax_k.set_xticks(SET_SIZES); ax_k.set_xticklabels([str(n) for n in SET_SIZES])
ax_k.set_xlabel('items'); ax_k.set_ylabel('kurtosis')
ax_k.text(-0.18,1.08,r'$\mathbf{f}$',transform=ax_k.transAxes,fontsize=15,fontweight='bold',va='top')

fig.suptitle('Bays (2014) Fig 2 — GP Model Predictions', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()